# Crop Disease Detection - Dataset Exploration
**ICS 3202: Artificial Intelligence - Project Deliverable 1**

**Group problem area:** Building a mobile application that lets a farmer photograph a crop leaf and receive an instant disease diagnosis, powered by an ML image classification engine.

**Dataset explored:** PlantVillage Dataset (Hughes & Salathé, 2015) maintained at [github.com/spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset).

## 1. Candidate open source datasets

The table below lists the open source datasets considered for this problem area, along with the variable in each that would serve as the **expected output (target label)** of the final application.

| # | Dataset | Description | Expected output variable |
|---|---------|-------------|---------------------------|
| 1 | **PlantVillage** ([spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset)) | 54,305 lab-condition leaf images, 14 crop species, 38 crop–disease classes | `disease_label` - the crop-disease (or healthy) class of the leaf |
| 2 | **New Plant Diseases Dataset** (Kaggle, augmented version of PlantVillage, ~87,000 images) | Offline-augmented version of PlantVillage, same 38 classes | `disease_label` - crop-disease class |
| 3 | **PlantDoc** (Singh et al., 2020) | ~2,598 real-world field images (non-lab backgrounds), 13 species, 27 classes | `disease_label` - crop-disease class, under real field conditions |
| 4 | **Cassava Leaf Disease Classification** (Kaggle / Makerere AI Lab, ~21,397 images) | Field images of cassava leaves collected in Uganda, 5 classes (4 diseases + healthy) | `disease_class` - cassava disease category |

### Why PlantVillage was selected
- It is the largest and most class balanced of the candidates, with 54,305 labeled images across 38 classes.
- It is the most widely benchmarked dataset in plant-disease-classification literature, which makes it easier to validate our model's performance against published results.
- The dataset was originally curated specifically to support the development of **mobile** disease diagnostic tools using machine learning  a direct match for our project's ML + Mobile App approach.

## 2. Fetching the dataset

The dataset is organized as one folder per crop-disease class under `raw/color`.

In [ ]:
# Clone only the raw/color portion of the PlantVillage repository
!git clone --filter=blob:none --no-checkout --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git
%cd PlantVillage-Dataset
!git sparse-checkout init --cone
!git sparse-checkout set raw/color
!git checkout master
%cd ..

## 3. Building the working dataframe

Since this is an image dataset, "rows" and "columns" refer to a metadata table we construct: one row per image, with columns describing that image (its file path, crop species, disease label, health status, file format, and size). This metadata table is what the exploration questions below are answered against.

In [ ]:
import os
import pandas as pd

DATA_DIR = "PlantVillage-Dataset/raw/color"

records = []
for class_folder in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_folder)
    if not os.path.isdir(class_path):
        continue
    if "___" in class_folder:
        crop, disease = class_folder.split("___", 1)
    else:
        crop, disease = class_folder, "unknown"
    for filename in os.listdir(class_path):
        filepath = os.path.join(class_path, filename)
        records.append({
            "filepath": filepath,
            "crop_species": crop,
            "disease_label": disease,
            "is_healthy": disease.lower() == "healthy",
            "file_extension": filename.split(".")[-1].lower(),
            "file_size_kb": round(os.path.getsize(filepath) / 1024, 2)
        })

df = pd.DataFrame(records)
df.head()

## 4. Data Exploration

### a) How many rows and columns are contained in the dataset?

In [ ]:
num_rows, num_cols = df.shape
print(f"Rows: {num_rows}")
print(f"Columns: {num_cols}")
df.shape

### b) What datatypes are contained in the dataset?

In [ ]:
df.dtypes

### c) Is the dataset complete? i.e. no missing values?

In [ ]:
missing = df.isnull().sum()
print(missing)
print()
is_complete = missing.sum() == 0
print(f"Dataset is complete (no missing values): {is_complete}")

### d) Slice out the first 15 rows and last 20 rows, merge into `df_sample`, and display it

In [ ]:
df_sample = pd.concat([df.head(15), df.tail(20)], ignore_index=True)
df_sample